In [ ]:
from typing import TypedDict,Annotated,Sequence
from langgraph.graph import StateGraph,END
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader,TextLoader
from langchain_core.documents import Document
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from langchain_community.vectorstores import FAISS
from langchain_classic.tools import Tool
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper
from langchain_classic.agents import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini")

class State(TypedDict):
    messages : Annotated[Sequence[BaseMessage],add_messages]

def make_retriever_tool_from_txt(file,name,desc):
    docs = TextLoader(file_path=file,encoding="utf-8").load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50).split_documents(docs)
    vs = FAISS.from_documents(documents=chunks)
    retriever = vs.as_retriever()

    def tool_func(query:str)->str:
        print(f"Using tool {name}")
        result = retriever.invoke(query)
        return "\n\n".join(doc.page_content for doc in result)
    
    return Tool(name=name,description=desc,func=tool_func)

internal_tool1 = make_retriever_tool_from_txt("internaldocs.txt","Internal_Doc","This doc is about transformer")

internal_tool2 = make_retriever_tool_from_txt("internaldocs_2.txt","Internal_Doc2","This doc is about agents")

def arxiv_query(query:str)->str:
    api_wrapper = ArxivAPIWrapper()
    arxiv = ArxivQueryRun(api_wrapper=api_wrapper)
    return arxiv.run(query)

arxiv_tool = Tool(name="Arxiv",func=arxiv_query,description="This tool is arxiv tool")

def wiki_query(query:str)->str:
   api_wrapper = WikipediaAPIWrapper()
   wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
   return wiki.run(query)

wiki_tool = Tool(name="Wiki",func=wiki_query,description="This took to search web")

Tools = [arxiv_tool,wiki_tool,internal_tool1,internal_tool2]

llm_with_tools = llm.bind_tools(tools=Tools)

react_node = create_react_agent(llm=llm,tools=Tools)

builder = StateGraph(State)

builder.add_node("")









